In [ ]:
# Importing Required Libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from text_preprocessing import preprocess_text  # Assuming this handles medical-specific preprocessing

# Step 1: Load Dataset
# Load the dataset (assuming it's a CSV file with columns: 'text_column' and 'label')
df = pd.read_csv('full_medical_dataset_5000.csv')

# Check the first few rows to ensure data format
print(df.head())

# Step 2: Preprocess the text using the text preprocessing module
# Assuming 'text_column' contains the medical text and 'label' contains the target labels
df['processed_text'] = df['text_column'].apply(preprocess_text)

# Step 3: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['processed_text'], df['label'], test_size=0.2, random_state=42)

# Step 4: Tokenize the text and convert to numerical format for the model
# Create a tokenizer, fitting it on the training text data
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

# Convert the text data into sequences of integers
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences for equal length (e.g., 100 words per sequence)
X_train_seq = tf.keras.preprocessing.sequence.pad_sequences(X_train_seq, padding='post', maxlen=100)
X_test_seq = tf.keras.preprocessing.sequence.pad_sequences(X_test_seq, padding='post', maxlen=100)

# Step 5: Build the Model
# Using an LSTM network for text classification
model = Sequential([
    Embedding(input_dim=5000, output_dim=128, input_length=100),
    LSTM(128),
    Dense(1, activation='sigmoid')  # For binary classification, use 'sigmoid'
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Step 6: Train the Model
history = model.fit(X_train_seq, y_train, epochs=10, validation_data=(X_test_seq, y_test))

# Step 7: Save the Model
model.save('saved_model/my_model.h5')

# Step 8: Evaluate the Model
loss, accuracy = model.evaluate(X_test_seq, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Step 9: Plot Training and Validation Accuracy
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.legend()
plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.show()

# Optionally, save evaluation metrics in a CSV or PNG
evaluation_metrics = {
    'loss': loss,
    'accuracy': accuracy
}
eval_df = pd.DataFrame([evaluation_metrics])
eval_df.to_csv('evaluation_metrics/results.csv', index=False)
